In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-4 ; m=1.0 ; a=2.0 ; g=9.81 ;                        # paraboloide  r² = a z  (z hacia arriba)
tf=8; N=int(tf/dt);

t=np.linspace(0,tf,N)


# Órbita circular de radio r_c (referencia para las pequeñas oscilaciones):  φ̇_c² = 2g/a ,  J_c = m r_c² φ̇_c
r_c   = 1.0
phd_c = np.sqrt(2*g/a)
J_c   = m*r_c**2*phd_c
om_r  = np.sqrt(8*g*a/(a**2 + 4*r_c**2))                  # frecuencia de las oscilaciones radiales alrededor de la órbita circular

# Condiciones iniciales  (r, ṙ, φ̇)   con  φ(0) = 0
def caso_peq(delta0):
    """Pequeñas oscilaciones radiales: r(0) = r_c + δ0, ṙ(0) = 0 y el mismo J que la órbita circular."""
    r0 = r_c + delta0
    return dict(label=f'$\\delta_0={delta0}$', r=r0, rd=0.0, phd=J_c/(m*r0**2))

# SITUACIÓN 1: movimiento cualquiera
caso_general = dict(label='movimiento general', r=1.2, rd=0.8, phd=1.5)

# SITUACIÓN 2: pequeñas oscilaciones alrededor de la órbita circular. Los otros δ0 se usan para comparar con la solución analítica
casos = [caso_peq(0.1), caso_peq(0.025), caso_peq(0.05), caso_peq(0.2), caso_general]
i_peq, i_gen = 0, 4                                         # índices de las dos situaciones en 'casos'

## FUNCIONES GENERALES: LAGRANGIANO, HAMILTONIANO, EULER-LAGRANGE (CON MULTIPLICADORES), HAMILTON Y PARÉNTESIS DE POISSON

In [ ]:
#FUNCIONES GENERALES (valen para cualquier sistema)
from types import SimpleNamespace

def lagrangian_hamiltonian(T, U, qs, qds, ps):
    """A partir de T y U devuelve:
         L = T − U                        (Lagrangiano)
         p_i = ∂L/∂q̇_i                    (momentos generalizados)
         H = Σ p_i q̇_i − L                (Hamiltoniano), SIEMPRE expresado en variables canónicas (q, p)
       T, U: expresiones de SymPy en (q, q̇) ;  qs, qds, ps: listas de símbolos (coordenadas, velocidades, momentos).
       Devuelve un objeto con L, p, H y, además, H_vel = H(q, q̇) y qdot = {q̇_i: q̇_i(q, p)}."""
    L = sp.simplify(T - U)
    p_expr = [sp.diff(L, qd) for qd in qds]                                           # p_i = ∂L/∂q̇_i  (derivadas parciales)
    H_vel  = sp.simplify(sum(p*qd for p, qd in zip(p_expr, qds)) - L)                 # H(q, q̇) = Σ p q̇ − L
    sol    = sp.solve([sp.Eq(P, pe) for P, pe in zip(ps, p_expr)], qds, dict=True)[0]  # se invierte p(q, q̇)  ->  q̇(q, p)
    H      = sp.simplify(H_vel.subs(sol))                                             # H(q, p)
    return SimpleNamespace(L=L, p=p_expr, H=H, H_vel=H_vel, qdot=sol)


def euler_lagrange(L, qs, qds, qdds, t=None):
    """Ecuaciones de Euler-Lagrange  d/dt(∂L/∂q̇_i) − ∂L/∂q_i = 0  para cualquier L(q, q̇ [, t]).
       qdds: símbolos para las aceleraciones q̈_i ; t: símbolo del tiempo si L depende explícitamente de él.
       Devuelve la lista de expresiones (cada una igualada a 0)."""
    n, EL = len(qs), []
    for i in range(n):
        p_i = sp.diff(L, qds[i])
        dp_dt = sum(sp.diff(p_i, qs[j])*qds[j] + sp.diff(p_i, qds[j])*qdds[j] for j in range(n))   # derivada total respecto a t
        if t is not None:
            dp_dt += sp.diff(p_i, t)
        EL.append(sp.simplify(dp_dt - sp.diff(L, qs[i])))
    return EL


def hamilton_equations(H, qs, ps):
    """Ecuaciones de Hamilton para cualquier H(q, p):  q̇_i = ∂H/∂p_i ,  ṗ_i = −∂H/∂q_i.  Devuelve (qdot_eq, pdot_eq)."""
    return [sp.diff(H, P) for P in ps], [-sp.diff(H, q) for q in qs]


def poisson_bracket(f, g, qs, ps):
    """Paréntesis de Poisson  {f, g} = Σ_i ( ∂f/∂q_i ∂g/∂p_i − ∂f/∂p_i ∂g/∂q_i )."""
    return sp.simplify(sum(sp.diff(f, q)*sp.diff(g, p) - sp.diff(f, p)*sp.diff(g, q) for q, p in zip(qs, ps)))


def euler_lagrange_multipliers(L, qs, qds, qdds, constraints, lams, t=None):
    """Ecuaciones de Euler-Lagrange con multiplicadores de Lagrange:
         d/dt(∂L/∂q̇_i) − ∂L/∂q_i − Σ_α λ_α ∂G_α/∂q_i = 0      con ligaduras G_α(q) = 0.
       constraints: lista de G_α ;  lams: lista de símbolos λ_α. Devuelve la lista de expresiones (cada una igualada a 0)."""
    EL = euler_lagrange(L, qs, qds, qdds, t)
    return [sp.simplify(EL[i] - sum(lam*sp.diff(G, qs[i]) for lam, G in zip(lams, constraints))) for i in range(len(qs))]

## (a) ECUACIONES DE MOVIMIENTO CON EL MULTIPLICADOR DE LAGRANGE

In [ ]:
#(a) Lagrangiano en cilíndricas (r, φ, z) y ecuaciones de movimiento con multiplicador de Lagrange
r_s, ph_s, z_s = sp.symbols('r phi z', real=True)
rd_s, phd_s, zd_s = sp.symbols('rdot phidot zdot', real=True)
rdd_s, phdd_s, zdd_s = sp.symbols('rddot phiddot zddot', real=True)
lam_s = sp.symbols('lambda', real=True)
m_, a_, g_ = sp.symbols('m a g', positive=True)

qs3, qds3, qdds3 = [r_s, ph_s, z_s], [rd_s, phd_s, zd_s], [rdd_s, phdd_s, zdd_s]
T3 = sp.Rational(1,2)*m_*(rd_s**2 + r_s**2*phd_s**2 + zd_s**2)
U3 = m_*g_*z_s
L3 = T3 - U3
E3 = T3 + U3
G  = a_*z_s - r_s**2                                        # ligadura  G(r, z) = a z − r² = 0

# d/dt(∂L/∂q̇_i) − ∂L/∂q_i = λ ∂G/∂q_i
EL3 = euler_lagrange_multipliers(L3, qs3, qds3, qdds3, [G], [lam_s])
for nm, e in zip(('r', 'φ', 'z'), EL3):
    print(f'EL_{nm}:  {e} = 0')
print('\np_r, p_φ, p_z =', [sp.diff(L3, qd) for qd in qds3])

# ¿Qué se conserva?
print('\n∂L/∂φ =', sp.diff(L3, ph_s), '  ->  φ cíclica:  p_φ = m r² φ̇ ≡ J se conserva')
print("p_r y p_z NO se conservan:  ∂L/∂r =", sp.diff(L3, r_s), ' ,  ṗ_z = −m g + λ a ≠ 0')
acc = sp.solve(EL3, qdds3, dict=True)[0]                      # q̈ en función de (q, q̇, λ)
dE  = sum(sp.diff(E3, q)*qd + sp.diff(E3, qd)*acc[qdd] for q, qd, qdd in zip(qs3, qds3, qdds3))
print('dE/dt =', sp.simplify(dE), '  = λ Ġ,   con  Ġ = a ż − 2 r ṙ = 0 sobre la ligadura  ->  E se conserva')

## (b) MULTIPLICADOR λ EN FUNCIÓN DE E Y J

In [ ]:
#(b) Multiplicador de Lagrange:  λ(r, ṙ, φ̇)  y expresión en función de E y J
# Ġ y G̈ sobre la ligadura:  G̈ = Σ ∂G/∂q_i q̈_i + Σ ∂²G/∂q_i∂q_j q̇_i q̇_j = 0
Gdd = (sum(sp.diff(G, qs3[i])*qdds3[i] for i in range(3))
       + sum(sp.diff(G, qs3[i], qs3[j])*qds3[i]*qds3[j] for i in range(3) for j in range(3)))
sol_lam   = sp.solve([EL3[0], EL3[2], Gdd], [rdd_s, zdd_s, lam_s], dict=True)[0]
lam_state = sp.simplify(sol_lam[lam_s])
print('λ(r, ṙ, φ̇) =', lam_state)

J_s, E_s = sp.symbols('J E', real=True)
E_surf   = sp.simplify(E3.subs({zd_s: 2*r_s*rd_s/a_, z_s: r_s**2/a_}))             # E sobre la superficie
lam_claim = (4*E_s/a_**2 + 8*J_s**2/(m_*a_**4) + m_*g_/a_)*(1 + 4*r_s**2/a_**2)**(-2)
res = sp.simplify(lam_claim.subs({E_s: E_surf, J_s: m_*r_s**2*phd_s}) - lam_state)
print('\nλ = (4E/a² + 8J²/(m a⁴) + m g/a)(1 + 4r²/a²)⁻²   con   J = p_φ = m r² φ̇  (momento angular respecto al eje z)')
print('   λ(enunciado) − λ(r, ṙ, φ̇) =', res, '   ->  demostrado' if res == 0 else '')
print('   (E ≥ 0 y el paréntesis es > 0  ->  λ > 0 siempre)')

## (c) FUERZA DE LIGADURA

In [ ]:
#(c) Fuerza de ligadura   F^ℓ = λ ∇G :   F_r = λ ∂G/∂r ,  F_φ = (λ/r) ∂G/∂φ ,  F_z = λ ∂G/∂z
F_r, F_ph, F_z = lam_s*sp.diff(G, r_s), lam_s*sp.diff(G, ph_s)/r_s, lam_s*sp.diff(G, z_s)
print('F^ℓ = (F_r, F_φ, F_z) =', (F_r, F_ph, F_z), '   ->  F^ℓ = λ (−2r e_r + a e_z)')

F_mod = sp.sqrt(F_r**2 + F_z**2)                                   # = |λ| sqrt(4r² + a²)
F_EJ  = sp.simplify(lam_claim*sp.sqrt(4*r_s**2 + a_**2))
print('|F^ℓ| =', F_mod, '   ->   |F^ℓ|(r) =', sp.simplify(F_EJ), '   (con λ > 0)')
print('   |F^ℓ|(r) = a (4E/a² + 8J²/(m a⁴) + m g/a) / (1 + 4r²/a²)^(3/2)   ->  decrece con r')

# ¿Es perpendicular al movimiento?   v = ṙ e_r + r φ̇ e_φ + ż e_z   ->   F·v = λ (−2 r ṙ + a ż) = λ Ġ
Fv = sp.simplify((F_r*rd_s + F_ph*r_s*phd_s + F_z*zd_s).subs(zd_s, 2*r_s*rd_s/a_))
print('F^ℓ · v sobre la superficie =', Fv, '   ->  F^ℓ ⟂ v (es normal a la superficie): no realiza trabajo')

## HAMILTONIANO SOBRE LA SUPERFICIE, ECUACIONES DE HAMILTON Y PARÉNTESIS DE POISSON

In [ ]:
#Hamiltoniano sobre la superficie (z = r²/a):  coordenadas (r, φ)   (con las funciones generales)
pr_s, pph_s = sp.symbols('p_r p_phi', real=True)
T_red = sp.simplify(T3.subs(zd_s, 2*r_s*rd_s/a_))
U_red = m_*g_*r_s**2/a_
E_red = T_red + U_red
qs, qds, ps = [r_s, ph_s], [rd_s, phd_s], [pr_s, pph_s]

lag = lagrangian_hamiltonian(T_red, U_red, qs, qds, ps)
L_sym, p_expr, H_can = lag.L, lag.p, lag.H
print('T =', T_red)
print('p =', p_expr)
print('H(r, p_r, p_φ) =', H_can)
print('H − E =', sp.simplify(lag.H_vel - E_red), '   ->  H = E' if sp.simplify(lag.H_vel - E_red) == 0 else '   ->  H ≠ E')
H_nice = pr_s**2/(2*m_*(1 + 4*r_s**2/a_**2)) + pph_s**2/(2*m_*r_s**2) + m_*g_*r_s**2/a_
print('H = p_r²/(2m(1+4r²/a²)) + p_φ²/(2 m r²) + m g r²/a  ->  diferencia con la expresión anterior =', sp.simplify(H_can - H_nice))

# Ecuaciones de Hamilton
qdot_eq, pdot_eq = hamilton_equations(H_can, qs, ps)
print('\nṙ =', qdot_eq[0], ' ,  φ̇ =', qdot_eq[1])
print('ṗ_r =', sp.simplify(pdot_eq[0]), ' ,  ṗ_φ =', pdot_eq[1])

# Paréntesis de Poisson
print('\n{q_i, p_j} =', sp.Matrix(2, 2, lambda i, j: poisson_bracket(qs[i], ps[j], qs, ps)).tolist())
for nm, f in (('H', H_can), ('p_r', pr_s), ('p_φ', pph_s)):
    val = poisson_bracket(f, H_can, qs, ps)
    print(f'{{{nm}, H}} = 0  ->  {nm} se conserva' if val == 0 else f'{{{nm}, H}} ≠ 0  ->  {nm} NO se conserva')

par_s = (m_, a_, g_)
f_num   = sp.lambdify((*qs, *ps, *par_s), qdot_eq + pdot_eq, 'numpy', cse=True)    # (ṙ, φ̇, ṗ_r, ṗ_φ)
H_num   = sp.lambdify((*qs, *ps, *par_s), H_can, 'numpy')
qd_num  = sp.lambdify((*qs, *ps, *par_s), [lag.qdot[qd] for qd in qds], 'numpy')
E_num   = sp.lambdify((*qs, *qds, *par_s), E_red, 'numpy')
L_num   = sp.lambdify((*qs, *qds, *par_s), L_sym, 'numpy')
lam_num = sp.lambdify((r_s, rd_s, phd_s, *par_s), lam_state, 'numpy')                 # λ(r, ṙ, φ̇)

## FUNCION PARA EL MOVIMIENTO (RUNGE-KUTTA 4)

In [ ]:
def F(y, pars):
    """Segundo miembro de las ecuaciones de Hamilton. y: (4, ncasos) = (r, φ, p_r, p_φ)."""
    d = f_num(*y, *pars)
    return np.array([np.broadcast_to(v, y.shape[1:]) for v in d])


def Motion(y0, pars, t):
    """Integra las ecuaciones de Hamilton con Runge-Kutta de orden 4 (varios casos a la vez)."""
    nv, nc = y0.shape
    N = len(t)
    y = np.zeros((nv, nc, N))
    y[:, :, 0] = y0

    for i in range(N - 1):
        dt = t[i+1] - t[i]
        yi = y[:, :, i]

        k1 = F(yi, pars)
        k2 = F(yi + 0.5*dt*k1, pars)
        k3 = F(yi + 0.5*dt*k2, pars)
        k4 = F(yi + dt*k3, pars)

        y[:, :, i+1] = yi + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

    return y


#Estado inicial: momentos  p = ∂L/∂q̇
ci = {kk: np.array([cs[kk] for cs in casos]) for kk in casos[0] if kk != 'label'}
y0 = np.array([ci['r'], np.zeros(len(casos)),
               m*(1 + 4*ci['r']**2/a**2)*ci['rd'],                  # p_r = m (1 + 4r²/a²) ṙ
               m*ci['r']**2*ci['phd']])                             # p_φ = m r² φ̇ = J

#Simulación
pars = (m, a, g)
y = Motion(y0, pars, t)
r, ph, pr, pph = y
rd, phd = qd_num(*y, *pars)
z = r**2/a                                                          # altura sobre la superficie

## ENERGÍA, MOMENTO ANGULAR Y DIAGRAMA DE FASES

In [ ]:
#Energía, Hamiltoniano y momento angular J
E = E_num(r, ph, rd, phd, *pars)                           # E = T + U
H = H_num(r, ph, pr, pph, *pars)                           # H(r, φ, p_r, p_φ)
L = L_num(r, ph, rd, phd, *pars)
J = pph                                                    # J = p_φ = m r² φ̇

print(f'max |H − E| / |E0|  (todos los casos) = {np.abs(H - E).max()/np.abs(E[:, 0]).max():.2e}')
print(f'max |H − H0| / |H0| (todos los casos) = {(np.abs(H - H[:, :1]).max(axis=1)/np.abs(H[:, 0])).max():.2e}   ->  E = H se conserva')
ig, ip = i_gen, i_peq
print(f'J varía {np.ptp(J[ig]):.1e} (caso general)  ->  J se conserva   |   p_r varía {np.ptp(pr[ig]):.3f}  ->  p_r NO se conserva')
print(f'caso general:  E = {E[ig, 0]:.4f} J ,  J = {J[ig, 0]:.4f} kg m²/s ,  r ∈ [{r[ig].min():.3f}, {r[ig].max():.3f}] m')

cols = {ip: 'tab:blue', ig: 'tab:red'}
nom  = {ip: 'pequeñas oscilaciones', ig: 'movimiento general'}

# --- Potencial efectivo y diagrama de fases radial (r, p_r) ---
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
rg = np.linspace(0.15, 1.15*r[[ip, ig]].max(), 500)
for i_ in (ip, ig):
    Veff = J[i_, 0]**2/(2*m*rg**2) + m*g*rg**2/a
    ax[0].plot(rg, Veff, color=cols[i_], lw=2, label=f'V_eff, J = {J[i_, 0]:.2f}  ({nom[i_]})')
    ax[0].axhline(E[i_, 0], color=cols[i_], ls='--', lw=1)
ax[0].scatter([r_c], [J_c**2/(2*m*r_c**2) + m*g*r_c**2/a], s=70, color='tab:green', zorder=5, label=f'órbita circular r_c = {r_c}')
ax[0].set(xlabel='r (m)', ylabel='V_eff (J)', title='Potencial efectivo  J²/(2mr²) + m g r²/a   (líneas: E)', ylim=(0, 1.4*E[:, 0].max())); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

pmax = 1.2*np.abs(pr[[ip, ig]]).max()
Rg, Pg = np.meshgrid(rg, np.linspace(-pmax, pmax, 400))
for i_ in (ip, ig):
    ax[1].contour(Rg, Pg, H_num(Rg, 0.0, Pg, J[i_, 0], *pars), levels=[E[i_, 0]], colors=cols[i_], linestyles='--', linewidths=1)
    ax[1].plot(r[i_], pr[i_], color=cols[i_], lw=2, label=nom[i_])
    ax[1].scatter(r[i_, 0], pr[i_, 0], s=60, color=cols[i_], edgecolors='black', zorder=5)
ax[1].set(xlabel='r (m)', ylabel='p_r (kg m/s)', title='Diagrama de fases radial (H = E, con J fijo)'); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# --- Conservación en el tiempo (caso general) ---
s = max(1, N // 20000)
fig, ax = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
ax[0].plot(t[::s], E[ig, ::s], color='tab:blue', lw=3, label='E = T + U')
ax[0].plot(t[::s], H[ig, ::s], 'r--', lw=1.5, label='H(r, p_r, p_φ)')
ax[0].plot(t[::s], L[ig, ::s], color='tab:purple', lw=1.2, label='L = T − U  (no se conserva)')
ax[0].set(ylabel='J', title='Caso general: E, H y L'); ax[0].legend(fontsize=8)
ax[1].plot(t[::s], J[ig, ::s], color='tab:green', lw=2, label='J = p_φ  (cte)')
ax[1].plot(t[::s], pr[ig, ::s], color='tab:orange', lw=2, label='p_r  (no se conserva)')
ax[1].set(ylabel='momentos', title='Momentos generalizados'); ax[1].legend(fontsize=8)
eps = 1e-18
ax[2].semilogy(t[::s], np.abs(H[ig, ::s] - H[ig, 0])/abs(H[ig, 0]) + eps, color='tab:red', lw=1, label='|H − H₀| / |H₀|  (integrador RK4)')
ax[2].semilogy(t[::s], np.abs(H[ig, ::s] - E[ig, ::s])/abs(E[ig, 0]) + eps, color='tab:green', lw=1, label='|H − E| / |E₀|  (redondeo)')
ax[2].set(xlabel='t (s)', ylabel='error relativo', title='Error numérico'); ax[2].legend(fontsize=8, loc='lower right')
for A in ax: A.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## λ Y FUERZA DE LIGADURA A LO LARGO DE LA TRAYECTORIA

In [ ]:
#(b), (c) Multiplicador λ y fuerza de ligadura a lo largo de la trayectoria
lam_n  = lam_num(r, rd, phd, *pars)                                           # λ(r, ṙ, φ̇) del cuaderno (sympy)
lam_an = (4*E/a**2 + 8*J**2/(m*a**4) + m*g/a)*(1 + 4*r**2/a**2)**(-2)         # fórmula del enunciado con E y J
Fmod   = lam_n*np.sqrt(4*r**2 + a**2)                                         # |F^ℓ| = λ sqrt(4r² + a²)

# Comprobación independiente (segunda ley de Newton): a λ = F_z = m z̈ + m g , con z̈ por derivación numérica
zdd = np.gradient(np.gradient(z, t, axis=1), t, axis=1)
lam_newton = (m*zdd + m*g)/a
sl = slice(5, -5)
print(f'max |λ_sympy − λ_enunciado| / max|λ| = {np.abs(lam_n - lam_an).max()/np.abs(lam_an).max():.2e}')
print(f'max |λ_enunciado − λ_Newton| / max|λ| = {np.abs(lam_an[:, sl] - lam_newton[:, sl]).max()/np.abs(lam_an).max():.2e}   (z̈ por diferencias finitas)')

# F·v = 0:  v = (ṙ, r φ̇, ż),  F = λ (−2r, 0, a)
zd = 2*r*rd/a
Fv = lam_n*(-2*r*rd + a*zd)
print(f'max |F·v| / (|F| |v|) = {np.abs(Fv/(Fmod*np.sqrt(rd**2 + (r*phd)**2 + zd**2))).max():.1e}   ->  F^ℓ ⟂ movimiento')

s = max(1, N // 5000)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(t[::s], lam_an[ig, ::s], color='tab:blue', lw=2.5, label='fórmula del enunciado (E, J, r)')
ax[0].plot(t[::s], lam_n[ig, ::s], 'r--', lw=1.2, label='λ(r, ṙ, φ̇)')
ax[0].plot(t[::s], lam_newton[ig, ::s], 'k:', lw=1.2, label='(m z̈ + m g)/a  (Newton)')
ax[0].set(xlabel='t (s)', ylabel='λ (N/m)', title='Multiplicador de Lagrange (caso general)'); ax[0].legend(fontsize=8)

rr = np.linspace(0, 1.3*r[[ip, ig]].max(), 400)
for i_ in (ip, ig):
    C = 4*E[i_, 0]/a**2 + 8*J[i_, 0]**2/(m*a**4) + m*g/a
    ax[1].plot(rr, a*C*(1 + 4*rr**2/a**2)**(-1.5), color=cols[i_], lw=2, label=f'{nom[i_]}:  E = {E[i_, 0]:.2f},  J = {J[i_, 0]:.2f}')
    ax[1].plot(r[i_, ::s], Fmod[i_, ::s], '.', color=cols[i_], ms=2)
ax[1].set(xlabel='r (m)', ylabel='|F^ℓ| (N)', title='|F^ℓ|(r) = a C (1 + 4r²/a²)^(-3/2)   (puntos: simulación)'); ax[1].legend(fontsize=8)

ax[2].plot(t[::s], Fmod[ig, ::s], color='tab:green', lw=2, label='|F^ℓ|')
ax[2].plot(t[::s], (m*g*np.ones_like(t))[::s], 'k:', lw=1, label='m g')
ax[2].set(xlabel='t (s)', ylabel='N', title='Módulo de la fuerza de ligadura (caso general)'); ax[2].legend(fontsize=8)
for A in ax: A.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## PEQUEÑAS OSCILACIONES ALREDEDOR DE LA ÓRBITA CIRCULAR

In [ ]:
#Pequeñas oscilaciones radiales alrededor de la órbita circular   r(t) ≈ r_c + δ0 cos(ω_r t)
#   órbita circular:  φ̇_c² = 2g/a ,  r_c⁴ = J² a / (2 m² g)        ω_r² = V_eff''(r_c)/M(r_c) = 8 g a / (a² + 4 r_c²)
print(f'r_c = {r_c} m ;  φ̇_c = sqrt(2g/a) = {phd_c:.4f} rad/s ;  J_c = {J_c:.4f} ;  ω_r = {om_r:.5f} rad/s  (T_r = {2*np.pi/om_r:.4f} s)')
print(f'ω_r/φ̇_c = 2/sqrt(1 + 4r_c²/a²) = {om_r/phd_c:.4f}')

delta0s  = ci['r'][:4] - r_c
delta    = r[:4] - r_c
delta_an = delta0s[:, None]*np.cos(om_r*t)[None, :]

print(f'\n{"δ0":>8}{"max|δ−δ_an|/δ0":>18}{"φ̇ medio (rad/s)":>18}')
for i_ in range(4):
    print(f'{delta0s[i_]:>8.3f}{np.abs(delta[i_] - delta_an[i_]).max()/delta0s[i_]:>18.2e}{np.mean(phd[i_]):>18.4f}')

s = max(1, N // 5000)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(t[::s], r[i_peq, ::s], lw=2, label='numérica')
ax[0].plot(t[::s], (r_c + delta_an[i_peq])[::s], 'r--', lw=1.2, label='r_c + δ0 cos(ω_r t)')
ax[0].axhline(r_c, color='k', ls=':', lw=1, label='r_c')
ax[0].set(xlabel='t (s)', ylabel='r (m)', title=f'Oscilaciones radiales (δ0 = {delta0s[i_peq]:.2f})'); ax[0].legend(fontsize=8)
ax[1].plot(t[::s], (delta[i_peq] - delta_an[i_peq])[::s], color='tab:green')
ax[1].set(xlabel='t (s)', ylabel='δ_num − δ_an', title='Diferencia (desfase por no linealidad)')
for A in ax: A.grid(alpha=0.3)
plt.tight_layout(); plt.show()


#Frecuencia radial para distintos radios de la órbita circular (vértice: r_c → 0  ->  ω_r → 2 φ̇_c)
rcs   = np.array([0.2, 0.5, 1.0, 1.5, 2.5])
phdc  = np.sqrt(2*g/a)*np.ones_like(rcs)
r0_s  = rcs + 0.01
y0_s  = np.array([r0_s, np.zeros_like(rcs), np.zeros_like(rcs), m*rcs**2*phdc])             # mismo J que la órbita circular de radio r_c
t_s   = np.arange(0, 10, 1e-3)
r_s_  = Motion(y0_s, pars, t_s)[0]

def omega_numerica(x, tt):
    """Frecuencia a partir de los cruces ascendentes de x(t) por cero (interpolación lineal)."""
    cr = np.where((x[:-1] < 0) & (x[1:] >= 0))[0]
    tc = tt[cr] - x[cr]*(tt[cr+1] - tt[cr])/(x[cr+1] - x[cr])
    return 2*np.pi/np.mean(np.diff(tc))

print(f'\n{"r_c":>6}{"ω_r teórica":>14}{"ω_r numérica":>15}{"ω_r/φ̇_c":>11}{"2/sqrt(1+4r_c²/a²)":>20}')
for j, rc_ in enumerate(rcs):
    om_t = np.sqrt(8*g*a/(a**2 + 4*rc_**2))
    om_n = omega_numerica(r_s_[j] - rcs[j], t_s)
    print(f'{rc_:>6.2f}{om_t:>14.4f}{om_n:>15.4f}{om_n/phdc[j]:>11.4f}{2/np.sqrt(1 + 4*rc_**2/a**2):>20.4f}')

## ANIMACIÓN (3D + dos planos 2D)

In [ ]:
#Animación: función que anima un caso (vista 3D + planos lateral x-z y cenital x-y) y se aplica a las dos situaciones iniciales
from IPython.display import display
plt.rcParams['animation.embed_limit'] = 100        # MB; el límite por defecto (20) se queda corto

interactivo = True     # True : figuras interactivas, se gira la vista 3D con el ratón mientras se anima (requiere ipympl)
                       # False: animación HTML (el ratón no gira la vista; usa 'rotar' para que la cámara gire sola)
elev, azim  = 20, -60  # ángulos iniciales de la vista 3D (grados)
rotar       = 0        # giro automático de la cámara (grados por segundo; 0 = fija). Funciona en los dos modos

T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 15       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 2.0      # duración de la estela (s); 0 = sin estela
mostrar_F = True   # dibuja la fuerza de ligadura (flecha verde, longitud ∝ |F^ℓ|)

if interactivo:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')      # backend interactivo:  pip install ipympl
    except Exception as e:
        interactivo = False
        print('Recordatorio para el modo interactivo (instalar ipympl).', e)


def animate_paraboloid(k_caso, titulo='', T_anim=T_anim, fps=fps, speed=speed, trail_s=trail_s, elev=elev, azim=azim, rotar=rotar):
    """Crea la figura y la animación del caso 'k_caso' (índice en 'casos'). Devuelve (fig, anim)."""
    step    = max(1, round(speed / (fps * dt)))
    idx     = np.arange(0, min(N, int(T_anim / dt)), step)
    trail_n = int(trail_s / (dt * step))

    ts = t[idx]
    ra, pha = r[k_caso, idx], ph[k_caso, idx]
    xs, ys, zs = ra*np.cos(pha), ra*np.sin(pha), ra**2/a                       # posición de la partícula
    Fa = Fmod[k_caso, idx]
    nx, ny, nz = -2*xs, -2*ys, a*np.ones_like(xs)                              # F^ℓ ∝ λ (−2x, −2y, a)
    nn = np.sqrt(nx**2 + ny**2 + nz**2)
    Lf = 0.45*r[k_caso].max()*Fa/Fmod[k_caso].max()                            # longitud de la flecha ∝ |F^ℓ|
    fx, fy, fz = xs + Lf*nx/nn, ys + Lf*ny/nn, zs + Lf*nz/nn

    # Límites y paraboloide
    r_max = 1.15*r[k_caso].max()
    z_top = r_max**2/a
    Rl    = 1.1*r_max
    z_lo, z_hi = -0.05*z_top, 1.12*z_top + Lf.max()
    phi = np.linspace(0, 2*np.pi, 100)

    fig = plt.figure(figsize=(14, 4.8))
    ax3 = fig.add_subplot(1, 3, 1, projection='3d')
    axs = fig.add_subplot(1, 3, 2)                     # plano lateral x-z
    axt = fig.add_subplot(1, 3, 3)                     # plano cenital x-y
    fig.subplots_adjust(left=0.02, right=0.98, wspace=0.35, top=0.86)
    fig.suptitle(titulo, fontsize=12, y=0.99)

    # --- Vista 3D: paraboloide ---
    ax3.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), zlim=(z_lo, z_hi), xlabel='x', ylabel='y', zlabel='z', title='Vista 3D')
    ax3.set_box_aspect((2*Rl, 2*Rl, z_hi - z_lo))
    ax3.grid(False)                                    # sin rejilla
    ax3.view_init(elev=elev, azim=azim)
    Rr, Pp = np.meshgrid(np.linspace(0, r_max, 40), np.linspace(0, 2*np.pi, 80))
    ax3.plot_surface(Rr*np.cos(Pp), Rr*np.sin(Pp), Rr**2/a, color='tab:blue', alpha=0.12, linewidth=0, shade=False)   # superficie
    ax3.plot(r_max*np.cos(phi), r_max*np.sin(phi), z_top*np.ones_like(phi), 'k-', lw=1.2)                              # borde
    rm = np.linspace(0, r_max, 60)
    for a_ in np.linspace(0, 2*np.pi, 13)[:-1]:
        ax3.plot(rm*np.cos(a_), rm*np.sin(a_), rm**2/a, color='tab:blue', lw=0.6, alpha=0.6)                         # meridianos
    ax3.plot([0, 0], [0, 0], [z_lo, z_hi], 'k:', lw=1)  # eje z
    ax3.plot([0], [0], [0], 'ko', ms=4)                 # vértice O

    # --- Plano lateral (x-z): parábola ---
    xp = np.linspace(-r_max, r_max, 200)
    axs.set(xlim=(-Rl, Rl), ylim=(z_lo, z_hi), aspect='equal', xlabel='x (m)', ylabel='z (m)', title='Plano lateral (x-z)')
    axs.plot(xp, xp**2/a, 'k-', lw=1.5); axs.fill_between(xp, xp**2/a, z_top, color='tab:blue', alpha=0.08)
    axs.axvline(0, color='k', ls=':', lw=1); axs.plot(0, 0, 'ko', ms=4); axs.grid(alpha=0.3)

    # --- Plano cenital (x-y) ---
    axt.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), aspect='equal', xlabel='x (m)', ylabel='y (m)', title='Plano cenital (x-y)')
    axt.plot(r_max*np.cos(phi), r_max*np.sin(phi), 'k-', lw=1.2)
    axt.plot(0, 0, 'ko', ms=4); axt.grid(alpha=0.3)

    # Artistas
    p3,  = ax3.plot([], [], [], 'o', color='tab:red', mec='black', ms=9)
    tr3, = ax3.plot([], [], [], '-', color='tab:red', lw=1, alpha=0.7)
    F3,  = ax3.plot([], [], [], '-', color='tab:green', lw=2)
    pS,  = axs.plot([], [], 'o', color='tab:red', mec='black', ms=9)
    trS, = axs.plot([], [], '-', color='tab:red', lw=1, alpha=0.7)
    FS,  = axs.plot([], [], '-', color='tab:green', lw=2)
    pT,  = axt.plot([], [], 'o', color='tab:red', mec='black', ms=9)
    trT, = axt.plot([], [], '-', color='tab:red', lw=1, alpha=0.7)
    time_txt = fig.text(0.02, 0.93, '')

    def update(k):
        j = max(0, k - trail_n)
        p3.set_data_3d([xs[k]], [ys[k]], [zs[k]]);  tr3.set_data_3d(xs[j:k+1], ys[j:k+1], zs[j:k+1])
        pS.set_data([xs[k]], [zs[k]]);              trS.set_data(xs[j:k+1], zs[j:k+1])
        pT.set_data([xs[k]], [ys[k]]);              trT.set_data(xs[j:k+1], ys[j:k+1])
        if mostrar_F:
            F3.set_data_3d([xs[k], fx[k]], [ys[k], fy[k]], [zs[k], fz[k]])
            FS.set_data([xs[k], fx[k]], [zs[k], fz[k]])
        if rotar != 0:                                 # solo si se pide giro automático (si no, se respeta la vista del ratón)
            ax3.view_init(elev=elev, azim=azim + rotar*ts[k])
        time_txt.set_text(f't = {ts[k]:.2f} s    r = {ra[k]:.3f} m    z = {zs[k]:.3f} m    |F| = {Fa[k]:.2f} N')
        return p3, tr3, F3, pS, trS, FS, pT, trT, time_txt

    anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=False)   # blit=False: necesario con ejes 3D
    return fig, anim


# Se aplica a las dos situaciones iniciales
situaciones = [(i_gen, 'Situación 1: movimiento general'),
               (i_peq, 'Situación 2: pequeñas oscilaciones radiales alrededor de la órbita circular')]
anims = [animate_paraboloid(k_caso, titulo) for k_caso, titulo in situaciones]      # se guardan las referencias para que no se detengan

if interactivo:
    plt.show()                                     # no cerrar las figuras: son las que se manejan con el ratón
else:
    for fig_, anim_ in anims:
        plt.close(fig_)
        display(HTML(anim_.to_jshtml()))